In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
train_labels = pd.read_csv("/kaggle/input/stanford-rna-3d-folding-2/train_labels.csv")
train_sequences = pd.read_csv("/kaggle/input/stanford-rna-3d-folding-2/train_sequences.csv")
validation_labels = pd.read_csv("/kaggle/input/stanford-rna-3d-folding-2/validation_labels.csv")
validation_sequences = pd.read_csv("/kaggle/input/stanford-rna-3d-folding-2/validation_sequences.csv")
test_sequences = pd.read_csv("/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv")
sample_submission = pd.read_csv("/kaggle/input/stanford-rna-3d-folding-2/sample_submission.csv")
msa_folder = "/kaggle/input/stanford-rna-3d-folding-2/msa/"
pdb_rna_folder = "/kaggle/input/stanford-rna-3d-folding-2/pdb_rna/"
extra_folder = "/kaggle/input/stanford-rna-3d-folding-2/extra/"

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/stanford-rna-3d-folding-2/train_labels.csv'

In [ ]:
train_labels

,ID,resname,resid,x_1,y_1,z_1,chain,copy
0,157D_1,C,1,4.843,-5.640,13.265,A,1
1,157D_2,G,2,3.385,-7.613,8.267,A,1
2,157D_3,C,3,2.158,-6.751,2.949,A,1
3,157D_4,G,4,2.669,-4.843,-1.773,A,1
4,157D_5,A,5,3.509,0.239,-4.045,A,1
...,...,...,...,...,...,...,...,...
7794966,9ZC8_1436,G,1436,244.609,370.637,308.076,A,2
7794967,9ZC8_1437,A,1437,246.970,365.282,307.269,A,2
7794968,9ZC8_1438,A,1438,249.310,360.899,309.337,A,2
7794969,9ZC8_1439,U,1439,250.751,358.343,313.544,A,2


In [ ]:
train_sequences

,target_id,sequence,temporal_cutoff,description,stoichiometry,all_sequences,ligand_ids,ligand_SMILES
0,4TNA,GCGGAUUUAGCUCAGUUGGGAGAGCGCCAGACUGAAGAUCUGGAGG...,1978-04-12,FURTHER REFINEMENT OF THE STRUCTURE OF YEAST T...,A:1,>4TNA_1|Chain A[auth A]|TRNAPHE|\nGCGGAUUUAGCU...,MG,[Mg+2]
1,6TNA,GCGGAUUUAGCUCAGUUGGGAGAGCGCCAGACUGAAGAUCUGGAGG...,1979-01-16,CRYSTAL STRUCTURE OF YEAST PHENYLALANINE T-RNA...,A:1,>6TNA_1|Chain A[auth A]|TRNAPHE|\nGCGGAUUUAGCU...,MG,[Mg+2]
2,1TRA,GCGGAUUUAGCUCAGUUGGGAGAGCGCCAGACUGAAGAUCUGGAGG...,1986-07-14,RESTRAINED REFINEMENT OF THE MONOCLINIC FORM O...,A:1,>1TRA_1|Chain A[auth A]|TRNAPHE|\nGCGGAUUUAGCU...,MG,[Mg+2]
3,1TN2,GCGGAUUUAGCUCAGUUGGGAGAGCGCCAGACUGAAGAUCUGGAGG...,1986-10-24,CRYSTALLOGRAPHIC AND BIOCHEMICAL INVESTIGATION...,A:1,>1TN2_1|Chain A[auth A]|TRNAPHE|\nGCGGAUUUAGCU...,MG;PB;SPM,[Mg+2];[Pb+2];C(CCNCCCN)CNCCCN
4,1TN1,GCGGAUUUAGCUCAGUUGGGAGAGCGCCAGACUGAAGAUCUGGAGG...,1987-01-15,CRYSTALLOGRAPHIC AND BIOCHEMICAL INVESTIGATION...,A:1,>1TN1_1|Chain A[auth A]|TRNAPHE|\nGCGGAUUUAGCU...,MG;PB;SPM,[Mg+2];[Pb+2];C(CCNCCCN)CNCCCN
...,...,...,...,...,...,...,...,...
5711,9HLZ,GUAAAAAAUUUAUAAGAAUAUGAUGUUGGUUCAGAUUAAGCGCUAA...,2025-12-17,"Translational activators Aep1, Aep2 and Atp25 ...",r:1;AA:1;t:1,>9HLZ_79|Chain AC[auth r]|15S mitochondrial rR...,GTP;MG,c1nc2c(n1[C@H]3[C@@H]([C@@H]([C@H](O3)CO[P@](=...
5712,9HM0,GUAAAAAAUUUAUAAGAAUAUGAUGUUGGUUCAGAUUAAGCGCUAA...,2025-12-17,Translational activator Aep3 in complex with m...,r:1;AA:1;t:1,>9HM0_77|Chain YB[auth r]|15S mitochondrial rR...,GTP;MG,c1nc2c(n1[C@H]3[C@@H]([C@@H]([C@H](O3)CO[P@](=...
5713,9HM4,AAUCCCAUCACCAUCUUCCAGGAGCGAGAUCCCUCCAAAAUCGGGG...,2025-12-17,Structure of tRNA bound Ba1Cas12a3,C:1;D:1;B:1,>9HM4_3|Chain C[auth C]|targetRNA|\nAAUCCCAUCA...,MG,[Mg+2]
5714,9I1W,AAACUUUCAACAACGGAUCUCUUGGUUCUGGCAUCGAUGAAGAACG...,2025-12-17,High resolution structure of the thermophilic ...,4:1;3:1;1:1;9:1,>9I1W_3|Chain C[auth 4]|5.8S RNA|\nAAACUUUCAAC...,K;MG;SPD;SPM;ZN,[K+];[Mg+2];C(CCNCCCN)CN;C(CCNCCCN)CNCCCN;[Zn+2]


In [ ]:
test_sequences

NameError: name 'test_sequences' is not defined